# 00.01 — Setup Environment

**Tujuan notebook ini**: pastikan environment kerja siap untuk eksperimen LLM vs SLM. Kalau semua cell di sini hijau, kamu siap lanjut.

Notebook ini **bukan tentang LLM/SLM** — ini "halo dunia" untuk semua tools.

Bisa dijalankan di **laptop lokal** maupun **Google Colab** — lihat section 0 untuk bedanya.

---

## Yang akan kita verifikasi

0. **Environment** — lokal vs Colab (bootstrap)
1. **Python** ≥ 3.10
2. **PyTorch** — terinstall versi CPU (BUKAN CUDA wheel)
3. **HuggingFace stack** — transformers, datasets, tokenizers
4. **llama-cpp-python** — engine quantization (opsional)
5. **OpenAI SDK** — kita pakai untuk Groq juga (OpenAI-compatible)
6. **API key** — dari `.env` (lokal) atau Colab Secrets (Colab)
7. **Sample SLM (SmolLM2-135M)** — bisa di-download & generate teks
8. **Sample LLM call (Groq llama-3.1-8b)** — API responsive

Kalau ada cell yang gagal, baca pesan error → cek `docs/pitfalls-cpu.md` → ulang.

## 0. Bootstrap — jalan di lokal & Colab

**Cell ini WAJIB dijalankan paling pertama di setiap notebook repo ini.** Fungsinya:

- Deteksi kamu jalan di **lokal** atau **Colab**.
- Di **Colab**: clone repo + install dependencies (runtime Colab selalu fresh).
- Di **lokal & Colab**: cari **root repo** (folder yang punya `requirements.txt`) dan taruh di `sys.path`, supaya `from utils...` jalan **apapun working directory kernel-nya**.

> Kenapa perlu cari root repo? VSCode & JupyterLab sering set working directory kernel ke **folder notebook** (mis. `00-setup-dan-tools/`), bukan root repo. Tanpa ini, `from utils...` akan `ModuleNotFoundError` walau di lokal.

> Di Colab, ganti `REPO_URL` dengan URL GitHub repo kamu **setelah** di-push.

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"  # <-- ganti setelah push
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

# Cari root repo (folder yang punya requirements.txt) lalu taruh di sys.path.
# Robust untuk working directory manapun (root repo / subfolder notebook / Colab).
repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break

assert repo_root is not None, (
    "Tidak ketemu root repo (folder dengan requirements.txt). "
    "Pastikan kamu jalankan notebook dari dalam repo llm-vs-slm-lab."
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"IN_COLAB  = {IN_COLAB}")
print(f"repo_root = {repo_root}")
print("OK — sys.path sudah di-set, `from utils...` siap dipakai.")

## 1. Cek versi Python

In [ ]:
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 10), "Disarankan Python >= 3.10"
print("OK — versi Python cukup baru.")

## 2. PyTorch — pastikan CPU-only

Kita SENGAJA pakai CPU. Kalau cell di bawah print `cuda_available=True`, berarti kamu install wheel CUDA — tidak masalah secara teknis, tapi makan disk 2GB tidak perlu.

Kalau `cuda_available=False`, perfect — itu yang kita mau. (Di Colab dengan runtime GPU, bisa True — itu OK juga, model kita kecil.)

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"cuda_available:  {torch.cuda.is_available()}")
print(f"mps_available:   {torch.backends.mps.is_available()}")  # Mac M-series

device = torch.device("cpu")
x = torch.randn(2, 3)
print(f"\nSmoke test tensor di CPU:\n{x}")
print("OK — PyTorch jalan.")

## 3. HuggingFace stack

Cek `transformers`, `datasets`, `tokenizers` ter-install. Belum kita download model di sini — itu di section 7.

In [ ]:
import transformers
import datasets
import tokenizers
import accelerate

print(f"transformers: {transformers.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"tokenizers:   {tokenizers.__version__}")
print(f"accelerate:   {accelerate.__version__}")

## 4. llama-cpp-python (opsional)

Kita pakai di notebook 02.03 untuk inference TinyLlama Q4 GGUF. Kalau install gagal di mesin mu, kamu bisa skip notebook itu — yang lain masih jalan.

Cell di bawah cuma cek import berhasil, BUKAN download model.

In [ ]:
try:
    import llama_cpp
    print(f"llama-cpp-python: {llama_cpp.__version__}")
    print("OK — bisa pakai notebook 02.03 nanti.")
except ImportError as exc:
    print(f"[WARN] llama-cpp-python belum ter-install: {exc}")
    print("Kamu bisa skip notebook 02.03; sisa repo tetap jalan.")
    print("Detail install: docs/pitfalls-cpu.md section 2.")

## 5. OpenAI SDK & dotenv

Kita pakai `openai` SDK untuk Groq juga (Groq OpenAI-compatible). `python-dotenv` baca `.env` file (relevan kalau jalan lokal; di Colab pakai Secrets, lihat section 6).

In [ ]:
import openai
from dotenv import load_dotenv

print(f"openai SDK: {openai.__version__}")
print("dotenv:     OK")

loaded = load_dotenv()
print(f"\nload_dotenv() → {loaded}")
print("  True  = file .env ketemu & ke-load (kamu jalan lokal dengan .env)")
print("  False = .env tidak ada (normal di Colab — kita pakai Secrets di section 6)")

## 6. Cek API key (GROQ_API_KEY)

Cara set key beda tergantung kamu jalan di mana:

**Di LOKAL (laptop):** pakai file `.env`
```bash
cp .env.example .env
# edit .env, isi GROQ_API_KEY=gsk_xxx
```

**Di GOOGLE COLAB:** pakai Colab Secrets (file `.env` TIDAK ikut saat clone karena di-gitignore)
1. Klik ikon kunci 🔑 di sidebar kiri Colab
2. Add new secret → Name: `GROQ_API_KEY`, Value: `gsk_xxx`
3. Aktifkan toggle **Notebook access**

Helper `get_secret()` di `utils/llm_clients.py` otomatis baca dari sumber yang benar — jadi cell yang sama jalan di dua-duanya.

Daftar key gratis (no credit card): https://console.groq.com/keys

In [ ]:
from utils.llm_clients import get_secret

groq_key = get_secret("GROQ_API_KEY")

if not groq_key:
    raise RuntimeError(
        "GROQ_API_KEY tidak ditemukan.\n"
        "Kalau di LOKAL:\n"
        "  1. Pastikan file .env ada di root repo (bukan .env.example)\n"
        "  2. Di dalam .env ada baris: GROQ_API_KEY=gsk_xxxxx\n"
        "  3. Restart kernel (Kernel → Restart), run ulang dari atas\n"
        "Kalau di COLAB:\n"
        "  1. Ikon kunci 🔑 → Add new secret → Name=GROQ_API_KEY\n"
        "  2. Aktifkan 'Notebook access'\n"
        "Daftar gratis: https://console.groq.com/keys"
    )

# jangan print key-nya — cuma confirm length nya
print(f"GROQ_API_KEY ditemukan ({len(groq_key)} karakter, mulai dengan '{groq_key[:4]}...')")

## 7. Smoke test SLM lokal — SmolLM2-135M

Download model paling kecil yang akan kita pakai (~270 MB). Pertama kali agak lama (download dari HuggingFace), berikutnya pakai cache lokal.

Kalau kamu tidak mau download dulu, kamu boleh **skip cell ini sekarang** dan jalankan saat butuh di notebook 02.02. Tapi disarankan jalankan sekarang biar tahu apakah download lancar di network mu.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

print(f"Downloading {model_id}... (pertama kali bisa 1–3 menit tergantung network)")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nLoaded. Total params: {n_params:,} (~{n_params / 1e6:.1f}M)")

In [ ]:
# generate teks pendek untuk smoke test
prompt = "Halo! Apa kabar?"
messages = [{"role": "user", "content": prompt}]

# Catatan kompatibilitas: di transformers 4.46+, apply_chat_template dengan
# return_tensors="pt" mengembalikan BatchEncoding (dict-like), BUKAN tensor.
# Pakai return_dict=True biar eksplisit, lalu **inputs untuk unpack ke generate().
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,  # greedy biar deterministik
        pad_token_id=tokenizer.eos_token_id,
    )

# potong bagian prompt; cuma ambil token yang di-generate
input_len = inputs["input_ids"].shape[1]
response = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
print(f"Prompt:   {prompt}")
print(f"Response: {response}")
print("\nOK — SmolLM2 jalan.")

Catatan: SmolLM2 itu kecil, jadi response nya untuk Bahasa Indonesia kadang aneh. Itu *fitur* repo ini — bagian dari pelajarannya: SLM kecil terbatas. Bandingkan dengan LLM di section berikut.

## 8. Smoke test LLM via Groq

Sekarang kita panggil LLM beneran via API. Model: `llama-3.1-8b-instant` (gratis di Groq).

Bandingkan latency dan kualitas response dengan SmolLM2 di atas.

In [ ]:
import time

from utils.llm_clients import GROQ_DEFAULT_MODEL, groq_client

client = groq_client()  # otomatis baca key dari .env (lokal) atau Secrets (Colab)

t0 = time.perf_counter()
resp = client.chat.completions.create(
    model=GROQ_DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Halo! Apa kabar?"}],
    max_tokens=40,
    temperature=0,
)
elapsed_ms = (time.perf_counter() - t0) * 1000

answer = resp.choices[0].message.content
usage = resp.usage

print(f"Response: {answer}")
print(f"\nLatency:        {elapsed_ms:.0f} ms")
print(f"Input tokens:   {usage.prompt_tokens}")
print(f"Output tokens:  {usage.completion_tokens}")
print(f"Model used:     {resp.model}")
print("\nOK — Groq API jalan.")

## Refleksi singkat

Sebelum lanjut, perhatikan dua angka ini:

- **SmolLM2-135M di laptop CPU**: butuh ~2–10 detik untuk 40 token.
- **Llama-3.1-8B di Groq**: butuh ~200–500 ms untuk 40 token, dan kualitas Bahasa nya jauh lebih natural.

Yang menarik: meski Groq pakai model **60x lebih besar**, latency nya **20x lebih cepat** karena hardware nya khusus (LPU). Tapi tentu saja: ada biaya per-token, dan kamu kirim data ke server mereka.

Tradeoff-tradeoff inilah yang akan kita bedah di modul-modul berikutnya.

## Selesai

Kalau semua cell di atas hijau:

- Environment siap (lokal atau Colab).
- API key terbaca.
- SLM lokal bisa di-load & generate.
- LLM via API responsive.

Lanjut ke **[../01-konsep-llm-vs-slm/01_apa_itu_language_model.ipynb](../01-konsep-llm-vs-slm/01_apa_itu_language_model.ipynb)** untuk konsep dasar.

Kalau ada error: copy pesan error, cek `docs/pitfalls-cpu.md`. Jawaban biasanya ada di sana.